# 02. RoPE와 causal multi-head attention

목표: 위치에 따른 vector 회전, scaled dot-product attention, causal mask와 multi-head projection을 직접 구현합니다.

In [ ]:
import numpy as np

rng = np.random.default_rng(21)
np.set_printoptions(precision=3, suppress=True)

## 1. Rotary Position Embeddings

짝을 이룬 좌표를 position별 각도로 회전합니다. 회전은 vector norm을 보존합니다.

In [ ]:
def apply_rope(x):
    seq_len, dim = x.shape
    if dim % 2:
        raise ValueError('RoPE dimension must be even')
    positions = np.arange(seq_len)[:, None]
    frequencies = 1.0 / (10000 ** (np.arange(0, dim, 2) / dim))
    angles = positions * frequencies[None, :]
    even, odd = x[:, 0::2], x[:, 1::2]
    rotated = np.empty_like(x)
    rotated[:, 0::2] = even * np.cos(angles) - odd * np.sin(angles)
    rotated[:, 1::2] = even * np.sin(angles) + odd * np.cos(angles)
    return rotated

vectors = rng.normal(size=(5, 8))
rotated = apply_rope(vectors)
print(rotated[:2])
assert np.allclose(np.linalg.norm(vectors, axis=1), np.linalg.norm(rotated, axis=1))
assert np.allclose(vectors[0], rotated[0])

## 2. Causal scaled dot-product attention

Upper triangle을 매우 작은 score로 바꿔 각 위치가 자신과 과거 위치만 보게 합니다.

In [ ]:
def softmax_rows(matrix):
    shifted = matrix - np.max(matrix, axis=-1, keepdims=True)
    exp_matrix = np.exp(shifted)
    return exp_matrix / exp_matrix.sum(axis=-1, keepdims=True)

def causal_attention(q, k, v):
    scores = q @ k.T / np.sqrt(q.shape[-1])
    future = np.triu(np.ones_like(scores, dtype=bool), k=1)
    masked_scores = np.where(future, -1e30, scores)
    weights = softmax_rows(masked_scores)
    return weights @ v, weights

sequence = rng.normal(size=(4, 8))
wq, wk, wv = [rng.normal(scale=0.2, size=(8, 4)) for _ in range(3)]
q = apply_rope(sequence @ wq)
k = apply_rope(sequence @ wk)
v = sequence @ wv
output, weights = causal_attention(q, k, v)
print('weights:')
print(weights)
assert output.shape == (4, 4)
assert np.allclose(weights.sum(axis=-1), 1.0)
assert np.allclose(np.triu(weights, k=1), 0.0)

## 3. Multi-head projection

각 head는 전체 input에서 독립적으로 투영된 Q/K/V를 사용합니다. Head 출력을 concatenate한 뒤 output matrix로 섞습니다.

In [ ]:
def multi_head_attention(x, num_heads=2):
    seq_len, d_model = x.shape
    if d_model % num_heads:
        raise ValueError('d_model must be divisible by num_heads')
    head_dim = d_model // num_heads
    head_outputs, head_weights = [], []
    projections = []
    for _ in range(num_heads):
        q_proj, k_proj, v_proj = [rng.normal(scale=0.2, size=(d_model, head_dim)) for _ in range(3)]
        projections.append((q_proj, k_proj, v_proj))
        q_head = apply_rope(x @ q_proj)
        k_head = apply_rope(x @ k_proj)
        v_head = x @ v_proj
        head_output, attention_map = causal_attention(q_head, k_head, v_head)
        head_outputs.append(head_output)
        head_weights.append(attention_map)
    concatenated = np.concatenate(head_outputs, axis=-1)
    output_projection = rng.normal(scale=0.2, size=(d_model, d_model))
    return concatenated @ output_projection, np.stack(head_weights), projections

mha_output, maps, projections = multi_head_attention(sequence, num_heads=2)
print('output:', mha_output.shape, 'attention maps:', maps.shape)
assert mha_output.shape == sequence.shape
assert maps.shape == (2, 4, 4)
assert all(qp.shape[0] == sequence.shape[1] for qp, _, _ in projections)

## 정리

RoPE는 Q/K comparison에 position을 반영하고, causal mask는 미래 정보 누출을 막습니다. Multi-head의 작은 공간은 원 vector의 고정 slice가 아니라 전체 표현에서 학습된 projection이라는 점이 중요합니다.